In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
# Run once in your environment:
!pip install sentence-transformers torch bert-score openpyxl

In [ ]:
import itertools
import numpy as np
import pandas as pd
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer
from bert_score import score as bert_score

In [ ]:
# ── 1. Texts to compare ──────────────────────────────────────────────────────
texts = {
    "Text_1": """...""",
    "Text_2": """...""",
    "Text_3": """...""",
    "Text_4": """...""",
    "Text_5": """...""",
    "Text_6": """...""",
    "Text_7": """...""",
    "Text_8": """...""",
    "Text_9": """...""",
    "Text_10": """...""",
}


labels = list(texts.keys())
text_list = list(texts.values())
n = len(text_list)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# METHOD 1 — SemScore (Cosine Similarity on Sentence Embeddings)
# ════════════════════════════════════════════════════════════════════════════════
# Reference: Aynetdinov & Akula, "SemScore: Automated Evaluation of Instruction-Tuned
#            LLMs based on Semantic Textual Similarity" (2024), arXiv:2401.17072
#            https://arxiv.org/abs/2401.17072
# Model hub: https://huggingface.co/sentence-transformers/all-mpnet-base-v2
# The paper recommends "all-mpnet-base-v2" as the best open-source encoder for
# computing semantic similarity between full-text passages.

# Symmetric: matrix[i][j] == matrix[j][i]
# Only compute upper triangle, then mirror it

model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
embeddings = model.encode(text_list, convert_to_tensor=True)

semscore_matrix = np.eye(n)  # diagonal = 1.0 (self-similarity)
for i in range(n):
    for j in range(i + 1, n):  # upper triangle only
        score = F.cosine_similarity(
            embeddings[i].unsqueeze(0),
            embeddings[j].unsqueeze(0)
        ).item()
        semscore_matrix[i][j] = score
        semscore_matrix[j][i] = score  # mirror

df_semscore = pd.DataFrame(semscore_matrix, index=labels, columns=labels)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# METHOD 2 — BERTScore (Token-level Contextual Similarity via BERT)
# ════════════════════════════════════════════════════════════════════════════════
# Reference: Zhang et al., "BERTScore: Evaluating Text Generation with BERT" (2020),
#            ICLR 2020, arXiv:1904.09675
#            https://arxiv.org/abs/1904.09675
# Library:   https://github.com/Tiiiger/bert_score
# Unlike SemScore which encodes the full passage into one vector, BERTScore computes
# pairwise cosine similarities between every token embedding of both texts, then
# derives Precision, Recall, and F1. This captures fine-grained semantic overlap
# and is more robust to paraphrase than n-gram methods.

# F1 is symmetric: F1[i][j] == F1[j][i]
# Precision[i][j] == Recall[j][i] (and vice versa)
# So only compute upper triangle; swap P/R to fill lower triangle

bertscore_P_matrix  = np.eye(n)  # diagonal = 1.0
bertscore_R_matrix  = np.eye(n)
bertscore_F1_matrix = np.eye(n)

for i in range(n):
    for j in range(i + 1, n):  # upper triangle only
        P, R, F1 = bert_score(
            [text_list[i]],  # candidate
            [text_list[j]],  # reference
            lang="en",
            verbose=False
        )
        p, r, f1 = P[0].item(), R[0].item(), F1[0].item()

        # Upper triangle: (i, j)
        bertscore_P_matrix[i][j]  = p
        bertscore_R_matrix[i][j]  = r
        bertscore_F1_matrix[i][j] = f1

        # Lower triangle: (j, i) — P and R are swapped, F1 is the same
        bertscore_P_matrix[j][i]  = r   # precision(j→i) == recall(i→j)
        bertscore_R_matrix[j][i]  = p   # recall(j→i)    == precision(i→j)
        bertscore_F1_matrix[j][i] = f1  # symmetric

df_bert_P  = pd.DataFrame(bertscore_P_matrix,  index=labels, columns=labels)
df_bert_R  = pd.DataFrame(bertscore_R_matrix,  index=labels, columns=labels)
df_bert_F1 = pd.DataFrame(bertscore_F1_matrix, index=labels, columns=labels)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# EXPORT — Save all matrices to a single Excel file (one sheet per metric)
# ════════════════════════════════════════════════════════════════════════════════
output_path = "similarity_results.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    df_semscore.round(4).to_excel(writer, sheet_name="SemScore")
    df_bert_P.round(4).to_excel(writer, sheet_name="BERTScore_Precision")
    df_bert_R.round(4).to_excel(writer, sheet_name="BERTScore_Recall")
    df_bert_F1.round(4).to_excel(writer, sheet_name="BERTScore_F1")

print(f"Results saved to {output_path}")